# Nemotron Reasoning — Colab A100 v3 (pushing towards 0.95)

**Target: 0.90+**. Realistic lift stack on top of v2:

| Stage | Mechanism | Expected lift |
|---|---|---|
| Base SFT (v3 recipe) | 15k synthetic (12 bit rules, 8 cipher rules, 10 algebra, 8 sequence families) + teacher CoT on *all* train.csv + self-verifying CoT template + wider LoRA targets + rsLoRA + NEFTune + packing + 5 epochs | 0.78–0.85 |
| Hard-case mining (RFT) | Adapter predicts on train, failing rows get teacher-verified CoT appended to dataset, re-train 2 more epochs | +3–5pp |
| DPO refinement | Build (chosen=teacher, rejected=adapter-wrong) pairs, run DPO with beta=0.1, lr=5e-6, 1 epoch | +1–3pp |

**Caveats.** 0.95 is not guaranteed — it depends heavily on the test set distribution. Sections below are modular; run Stage 1 first, check Kaggle, then add Stage 2, then Stage 3.

**Cost.** Teacher CoT with GPT-4o on the full train.csv + RFT + DPO can run $20–60 in OpenAI credits. Use `gpt-4o-mini` for cheap first pass.

## Setup

In [ ]:
import os, subprocess, sys, zipfile, getpass
from pathlib import Path
from google.colab import files

WORK_ROOT = Path("/content/project").resolve()
WORK_ROOT.mkdir(parents=True, exist_ok=True)

print("Upload udacity_upload.zip ...")
for name in files.upload():
    if name.endswith(".zip"):
        with zipfile.ZipFile(name) as zf: zf.extractall(WORK_ROOT)
        print("extracted", name)

data_dir = WORK_ROOT / "data"
(data_dir / "reports").mkdir(parents=True, exist_ok=True)
(data_dir / "synthetic").mkdir(parents=True, exist_ok=True)
if not (data_dir / "train.csv").is_file():
    print("Upload train.csv ...")
    for n in files.upload(): Path(n).rename(data_dir / n)

os.chdir(WORK_ROOT); sys.path.insert(0, str(WORK_ROOT))
assert (WORK_ROOT / "scripts" / "03_train_lora.py").is_file()
print("cwd:", os.getcwd())

## Config

In [ ]:
import torch
MODEL_ID = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
HF_TOKEN = os.environ.get("HF_TOKEN", "") or getpass.getpass("HF token (or blank): ")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "") or getpass.getpass("OpenAI key: ")
if OPENAI_API_KEY: os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# Data scale — push hard
SYNTHETIC_PER_KIND = 3500      # x4 types = 14k synthetic
MAX_PER_TYPE = 6000
MAX_TRAIN_CSV_FOR_COT = 0      # 0 = all rows (use gpt-4o or gpt-4o-mini)
COT_MODEL = "gpt-4o"           # best teacher; swap to gpt-4o-mini for cheap first pass

# Stage 1 training
TRAIN_MAX_SEQ = 4096
TRAIN_BATCH = 2
GRAD_ACCUM = 8       # effective batch 16
NUM_EPOCHS = 5.0
LR = 2e-4
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05

# Stage 2 hard-case mining
HARD_MINE_SAMPLES = 1000
REFINE_EPOCHS = 2.0

# Stage 3 DPO
DPO_PAIRS_SAMPLES = 1000
DPO_EPOCHS = 1.0
DPO_LR = 5e-6

print("GPU:", torch.cuda.get_device_name(0),
      f"({torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB)")

## Install deps

In [ ]:
import re
def pip_install(*a): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *a])
def pip_try(*a):
    try: pip_install(*a); return True
    except subprocess.CalledProcessError: return False

pip_install("-U", "pip", "setuptools", "wheel")
pip_install(
    "transformers>=4.45,<5", "peft>=0.12", "trl>=0.12", "datasets",
    "accelerate>=0.34", "bitsandbytes>=0.43", "psutil", "pandas", "numpy",
    "scikit-learn", "tqdm", "huggingface_hub", "ninja", "openai>=1.30",
)

_tf = torch.__version__
_mm = re.match(r"(\d+\.\d+)", _tf).group(1)
_cu = "cu12" if "cu12" in _tf else "cu11"
_py = f"cp{sys.version_info.major}{sys.version_info.minor}"
_abi = ["cxx11abiTRUE", "cxx11abiFALSE"] if int(_mm.split('.')[1]) >= 7 else ["cxx11abiFALSE", "cxx11abiTRUE"]
for pkg, tags in [
    ("causal-conv1d", [("v1.6.1","causal_conv1d","1.6.1"),("v1.5.4","causal_conv1d","1.5.4")]),
    ("mamba-ssm",     [("v2.3.1","mamba_ssm","2.3.1"),("v2.2.4","mamba_ssm","2.2.4")]),
]:
    gh = "https://github.com/Dao-AILab/causal-conv1d/releases/download" if "causal" in pkg else "https://github.com/state-spaces/mamba/releases/download"
    ok = False
    for tag, wn, wv in tags:
        if ok: break
        for a in _abi:
            if pip_try(f"{gh}/{tag}/{wn}-{wv}+{_cu}torch{_mm}{a}-{_py}-{_py}-linux_x86_64.whl"):
                ok = True; break
    if not ok:
        os.environ["CAUSAL_CONV1D_FORCE_BUILD"]="TRUE"; os.environ["MAMBA_FORCE_BUILD"]="TRUE"
        pip_install("--no-build-isolation", "--no-deps", pkg)

pip_try("--prefer-binary", "nvidia-cutlass-dsl>=4.4", "nvidia-cutlass-dsl-libs-base>=4.4")

## Download base model + auto-patch

In [ ]:
from huggingface_hub import login, snapshot_download
if HF_TOKEN: login(token=HF_TOKEN, add_to_git_credential=False)
MODEL_PATH = snapshot_download(MODEL_ID, resume_download=True)
print("model at", MODEL_PATH)

import glob, shutil
for mf in set(glob.glob("/root/.cache/huggingface/hub/**/modeling_nemotron_h.py", recursive=True)
             + glob.glob("/root/.cache/huggingface/modules/**/modeling_nemotron_h.py", recursive=True)):
    lines = Path(mf).read_text().splitlines(True); changed = False
    for i, l in enumerate(lines):
        if "final_hidden_states.index_add_(0, token_indices, weighted_output)" in l and "weighted_output.to(" not in l:
            lines[i] = l.replace("weighted_output)", "weighted_output.to(final_hidden_states.dtype))"); changed=True
        if ".to(expert_dtype)" in l:
            lines[i] = l.replace(".to(expert_dtype)", ".to(torch.bfloat16)"); changed=True
    if changed:
        Path(mf).write_text("".join(lines))
        pc = os.path.dirname(mf)+"/__pycache__"
        if os.path.exists(pc): shutil.rmtree(pc)
        print("patched", mf)

## Phase 1 — EDA

In [ ]:
subprocess.run([sys.executable, "scripts/01_eda.py",
                "--data-dir", "data", "--report-dir", "data/reports",
                "--tokenizer-model", str(MODEL_PATH)], check=True)

## Phase 2 — Massive synthetic + full-train.csv teacher CoT (Stage-1 data)

In [ ]:
cot_args = ["--cot-backend", "openai", "--cot-model", COT_MODEL, "--cot-max-tokens", "4096"]
if MAX_TRAIN_CSV_FOR_COT > 0: cot_args += ["--limit-train", str(MAX_TRAIN_CSV_FOR_COT)]
cmd = [sys.executable, "scripts/02_prepare_data.py",
       "--data-dir", "data", "--synthetic-dir", "data/synthetic",
       "--output", "data/train_sft.jsonl",
       "--tokenizer-model", str(MODEL_PATH),
       "--synthetic-per-kind", str(SYNTHETIC_PER_KIND),
       "--max-per-type", str(MAX_PER_TYPE),
       "--max-tokens-per-example", "6000"] + cot_args
print(" ".join(cmd)); subprocess.run(cmd, check=True)

## Stage 1 — LoRA SFT (wider targets, rsLoRA, packing, NEFTune, 5 epochs)

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env = os.environ.copy(); env["TOKENIZERS_PARALLELISM"]="false"
env["NEMOTRON_KAGGLE_PATCHES"]="0"; env["PYTHONUNBUFFERED"]="1"

cmd = [sys.executable, "scripts/03_train_lora.py",
       "--data-path", "data/train_sft.jsonl",
       "--output-dir", "lora_adapter_sft",
       "--checkpoint-dir", "lora_output_sft",
       "--model-path", str(MODEL_PATH),
       "--lora-target-mode", "kaggle_nemotron",
       "--lora-r", str(LORA_R),
       "--lora-alpha", str(LORA_ALPHA),
       "--lora-dropout", str(LORA_DROPOUT),
       "--batch-size", str(TRAIN_BATCH),
       "--grad-accum", str(GRAD_ACCUM),
       "--epochs", str(NUM_EPOCHS),
       "--lr", str(LR),
       "--max-seq-length", str(TRAIN_MAX_SEQ),
       "--warmup-ratio", "0.05",
       "--max-grad-norm", "1.0",
       "--neftune-alpha", "5.0",
       "--packing",
       "--force-peft", "--no-nemotron-kaggle-patches", "--dataloader-workers", "0"]
print(" ".join(cmd), flush=True)
proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout: print(line, end="", flush=True)
rc = proc.wait()
if rc != 0: raise subprocess.CalledProcessError(rc, cmd)

### Submit Stage-1 adapter (checkpoint to Kaggle before proceeding)

In [ ]:
subprocess.run([sys.executable, "scripts/05_package_submission.py",
                "--adapter-dir", "lora_adapter_sft",
                "--output", "submission_stage1.zip"], check=True)
from google.colab import files
files.download("submission_stage1.zip")

## Stage 2 — Hard-case mining + SFT round 2

Adapter answers `HARD_MINE_SAMPLES` rows from train.csv; wrong rows get teacher-verified CoT appended to `train_sft.jsonl`; retrain for `REFINE_EPOCHS` more.

In [ ]:
cmd = [sys.executable, "scripts/06_mine_hard.py",
       "--adapter-path", "lora_adapter_sft",
       "--base-model", str(MODEL_PATH),
       "--train-csv", "data/train.csv",
       "--append-to", "data/train_sft.jsonl",
       "--max-samples", str(HARD_MINE_SAMPLES),
       "--cot-backend", "openai", "--cot-model", COT_MODEL]
print(" ".join(cmd)); subprocess.run(cmd, check=True)

cmd = [sys.executable, "scripts/03_train_lora.py",
       "--data-path", "data/train_sft.jsonl",
       "--output-dir", "lora_adapter_sft2",
       "--checkpoint-dir", "lora_output_sft2",
       "--model-path", str(MODEL_PATH),
       "--lora-target-mode", "kaggle_nemotron",
       "--lora-r", str(LORA_R), "--lora-alpha", str(LORA_ALPHA),
       "--lora-dropout", str(LORA_DROPOUT),
       "--batch-size", str(TRAIN_BATCH), "--grad-accum", str(GRAD_ACCUM),
       "--epochs", str(REFINE_EPOCHS), "--lr", str(LR / 2),  # lower LR on refine
       "--max-seq-length", str(TRAIN_MAX_SEQ),
       "--warmup-ratio", "0.02", "--max-grad-norm", "1.0",
       "--neftune-alpha", "5.0", "--packing",
       "--force-peft", "--no-nemotron-kaggle-patches", "--dataloader-workers", "0"]
proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout: print(line, end="", flush=True)
if proc.wait(): raise SystemExit("stage 2 SFT failed")

subprocess.run([sys.executable, "scripts/05_package_submission.py",
                "--adapter-dir", "lora_adapter_sft2",
                "--output", "submission_stage2.zip"], check=True)
from google.colab import files
files.download("submission_stage2.zip")

## Stage 3 — DPO refinement

Build preference pairs (chosen=teacher, rejected=adapter-wrong) then run DPO on top of Stage-2 adapter.

In [ ]:
cmd = [sys.executable, "scripts/06b_build_dpo_pairs.py",
       "--adapter-path", "lora_adapter_sft2",
       "--base-model", str(MODEL_PATH),
       "--train-csv", "data/train.csv",
       "--output", "data/dpo_pairs.jsonl",
       "--max-samples", str(DPO_PAIRS_SAMPLES),
       "--cot-backend", "openai", "--cot-model", COT_MODEL]
print(" ".join(cmd)); subprocess.run(cmd, check=True)

cmd = [sys.executable, "scripts/07_dpo.py",
       "--pairs", "data/dpo_pairs.jsonl",
       "--sft-adapter", "lora_adapter_sft2",
       "--base-model", str(MODEL_PATH),
       "--output-dir", "lora_adapter_dpo",
       "--checkpoint-dir", "dpo_output",
       "--epochs", str(DPO_EPOCHS),
       "--lr", str(DPO_LR),
       "--batch-size", "1",
       "--grad-accum", "8",
       "--beta", "0.1",
       "--max-length", str(TRAIN_MAX_SEQ)]
proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout: print(line, end="", flush=True)
if proc.wait(): raise SystemExit("DPO failed")

subprocess.run([sys.executable, "scripts/05_package_submission.py",
                "--adapter-dir", "lora_adapter_dpo",
                "--output", "submission_stage3.zip"], check=True)
from google.colab import files
files.download("submission_stage3.zip")

## Notes

- Submit after each stage — if Stage 2 regresses, keep Stage 1 adapter as the submission.
- If Stage 3 DPO destabilizes (eval loss spikes), lower `DPO_LR` to 1e-6 and re-run; DPO is sensitive.
- Beyond Stage 3: run another round of hard-case mining with the DPO adapter; each extra RFT round typically gives diminishing returns (+0.5–1.5pp).
- If Kaggle pipeline permits `n>1` sampling + majority vote at inference time, self-consistency will add another 1–3pp. Most Kaggle competitions lock this to greedy — adapter cannot change that.
